# 🖐️ 손동작 탐지 모델 학습 (Colab)
모션인식 탐색 키오스크 — 🅐 모델 파트(성민수)용. **런타임 → 런타임 유형 변경 → T4 GPU** 후 위에서부터 실행하세요.

결과(가중치·평가 json)는 공유 드라이브 `motion_search_kiosk/` 에 저장됩니다. 공유 드라이브가 '공유 문서함'에 있으면 먼저 **내 드라이브에 바로가기 추가** 하세요.

In [ ]:
# 1) 구글 드라이브 연결
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/motion_search_kiosk'   # 바로가기 경로가 다르면 수정
!ls "$DRIVE"

In [ ]:
# 2) 저장소 받기 + 패키지 설치
!git clone https://github.com/jangjunwon-dev/motion_search_kiosk.git /content/motion_search_kiosk
%cd /content/motion_search_kiosk
!pip -q install -r requirements-model.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 3) 데이터 준비
- **이미 변환본이 드라이브에 있으면** (`data/processed/hagrid_v1`) 3-b 만 실행
- 처음이면 HaGRID 를 받아 3-a 로 변환 (전체는 너무 크므로 6개 클래스 경량판/샘플만)

In [ ]:
# 3-a) 원본 → 변환 (HaGRID 원본을 $DRIVE/data/raw/hagrid 에 올려둔 경우)
RAW = f'{DRIVE}/data/raw/hagrid'
!python training/prepare_hagrid.py --ann-dir "$RAW/annotations" --images-root "$RAW/images" \
    --out /content/hagrid_v1 --classes one fist palm like dislike peace \
    --max-per-class 3000 --format both --copy
!cp -r /content/hagrid_v1 "$DRIVE/data/processed/"

In [ ]:
# 3-b) 드라이브의 변환본을 로컬 디스크로 복사 (드라이브에서 직접 학습하면 매우 느림)
!rsync -a --info=progress2 "$DRIVE/data/processed/hagrid_v1/" /content/hagrid_v1/
# data.yaml 의 path 를 로컬 경로로 고쳐쓰기
import re
import pathlib
p = pathlib.Path('/content/hagrid_v1/yolo/data.yaml')
p.write_text(re.sub(r'^path: .*$', 'path: /content/hagrid_v1/yolo', p.read_text(), flags=re.M))
print(p.read_text())

In [ ]:
# 4) YOLO26n 학습 (T4 기준 3000장×6클래스, 50 epoch 약 1시간 내외 — 실제 시간은 기록해 두기)
!python training/train_yolo.py --data /content/hagrid_v1/yolo/data.yaml \
    --model yolo26n.pt --epochs 50 --imgsz 640 --batch 32 --name gesture_yolo26n

In [ ]:
# 5) 평가 (정확도 + 속도)
!python training/evaluate.py map --backend yolo --model models/gesture_yolo26n.pt \
    --data /content/hagrid_v1/yolo/data.yaml
!python training/evaluate.py speed --backend yolo --model models/gesture_yolo26n.pt \
    --images /content/hagrid_v1/yolo/images/test --n 200

In [ ]:
# 6) 결과를 드라이브에 저장
!mkdir -p "$DRIVE/models" "$DRIVE/발표·보고서/실험결과"
!cp models/*.pt "$DRIVE/models/"
!cp -r runs/detect/gesture_yolo26n "$DRIVE/발표·보고서/실험결과/"   # 학습 곡선·혼동행렬 그림 (PPT 용)
!cp runs/eval/*.json "$DRIVE/발표·보고서/실험결과/"
print('✅ 저장 완료 → models/README.md · docs/experiments.md 표 갱신하기')

## (비교 실험) RF-DETR
RF-DETR 는 메모리를 많이 써서 T4 에서는 batch 4 + grad_accum 4 권장.

In [ ]:
!python training/train_rfdetr.py --dataset /content/hagrid_v1/coco --size base \
    --epochs 20 --batch 4 --grad-accum 4 --output runs/rfdetr/gesture_rfdetr_base
!python training/evaluate.py map --backend rfdetr --model models/gesture_rfdetr_base.pth \
    --coco /content/hagrid_v1/coco/test
!python training/evaluate.py speed --backend rfdetr --model models/gesture_rfdetr_base.pth \
    --images /content/hagrid_v1/coco/test --n 200
!cp models/*.pth "$DRIVE/models/"; cp runs/eval/*.json "$DRIVE/발표·보고서/실험결과/"